In [ ]:
## Imports
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

Load Prediction Data

In [ ]:
# Load Prediction Data
base_directory = './'  # Replace with the actual path
predicted_vals = {}

for filename in os.listdir(base_directory):
    if filename.endswith(".csv"):
        geohash = filename.split('.')[0]

        df = pd.read_csv(os.path.join(base_directory, filename))

        predicted_vals[geohash] = df

# To check the data
for geohash, data in predicted_vals.items():
    print(f"Geohash: {geohash}")
    predicted, actual = data
    print(f"Predicted: {predicted[:5]}")  # Print first 5 values
    print(f"Actual: {actual[:5]}")

Function Definitions

In [ ]:
# Filtering function, used in plotting
def filter_outliers(actual, predicted, threshold=None, percentage=None):
    if threshold is not None:
        # Create a mask to filter out values greater than the threshold
        mask = actual <= threshold
    elif percentage is not None:
        # Calculate the percentage threshold based on actual values
        limit = np.percentile(actual, 100 - percentage)
        mask = actual <= limit
    else:
        # If neither threshold nor percentage is provided, return the original data
        return actual, predicted

    # Apply the mask to both actual and predicted values
    actual_filtered = actual[mask]
    predicted_filtered = predicted[mask]

    return actual_filtered, predicted_filtered

In [ ]:
# Plotting function: actual vs. predicted and percent error
def plot_results(predicted_vals, geohash, sample_size, plot_type='line', marker_color='blue', marker_size=10):
    # Check if the geohash is in the dictionary
    if geohash not in predicted_vals:
        print(f"Geohash {geohash} not found in the predictions dictionary.")
        return

    # Extract the predicted and actual values for the specified geohash
    predicted, actual = predicted_vals[geohash]

    # Flatten the arrays to 1D for plotting
    predicted = predicted.flatten()
    actual = actual.flatten()

    # Ensure the sample size is not greater than the filtered data size
    sample_size = min(sample_size, len(actual))

    # Calculate percentage error
    percentage_error = 100 * abs((actual - predicted) / actual)

    # Plot True Values vs Predictions
    plt.figure(figsize=(20, 15))
    if plot_type == 'line':
        plt.plot(actual[:sample_size], label='True Values', linestyle='-', linewidth=marker_size)
        plt.plot(predicted[:sample_size], label='Predictions', linestyle='-', linewidth=marker_size)
    elif plot_type == 'scatter':
        plt.scatter(range(sample_size), actual[:sample_size], label='True Values', c='blue', s=marker_size)
        plt.scatter(range(sample_size), predicted[:sample_size], label='Predictions', c=marker_color, s=marker_size)
    plt.legend()
    plt.xlabel('Sample Index')
    plt.ylabel('Time to Next Ride (s)')
    plt.title(f'True Values vs. Predictions for Geohash {geohash} (Sample Size: {sample_size})')
    plt.show()

    # Plot Percentage Error
    plt.figure(figsize=(20, 15))
    if plot_type == 'line':
        plt.plot(percentage_error[:sample_size], label='Percentage Error', linestyle='-', linewidth=marker_size)
    elif plot_type == 'scatter':
        plt.scatter(range(sample_size), percentage_error[:sample_size], label='Percentage Error', c=marker_color, s=marker_size)
    plt.legend()
    plt.xlabel('Sample Index')
    plt.ylabel('Percentage Error (%)')
    plt.title(f'Percentage Error for Geohash {geohash} (Sample Size: {sample_size})')
    plt.show()

In [ ]:
## Function to calculate metrics between a baseline(avg) and predicted
def calculate_performance_metrics(predicted_vals, geohash):
    # Check
    if geohash not in predicted_vals:
        print(f"Geohash {geohash} not found in the predictions dictionary.")
        return

    # Extract the predicted and actual values for the specified geohash
    predicted, actual = predicted_vals[geohash]

    # Flatten the arrays
    predicted = predicted.flatten()
    actual = actual.flatten()

    # Baseline predictions: using the mean of the actual values
    baseline_predicted = np.full_like(actual, np.mean(actual))

    # Calculate metrics for the model
    mae_model = mean_absolute_error(actual, predicted)
    mse_model = mean_squared_error(actual, predicted)
    rmse_model = np.sqrt(mse_model)


    # Calculate metrics for the baseline
    mae_baseline = mean_absolute_error(actual, baseline_predicted)
    mse_baseline = mean_squared_error(actual, baseline_predicted)
    rmse_baseline = np.sqrt(mse_baseline)

    return mae_model, mse_model, rmse_model, mae_baseline, mse_baseline, rmse_baseline

In [ ]:
# Plotting function: actual vs. predicted vs. baseline and percent error
def plot_filtered_results_with_baseline(predicted_vals, geohash, sample_size, plot_type='line', marker_color='blue', marker_size=10, threshold=None, save_path=None):
    # Check if the geohash is in the dictionary
    if geohash not in predicted_vals:
        print(f"Geohash {geohash} not found in the predictions dictionary.")
        return

    # Extract the predicted and actual values for the specified geohash
    predicted, actual = predicted_vals[geohash]

    # Flatten the arrays to 1D for plotting
    predicted = predicted.flatten()
    actual = actual.flatten()

    # Filter outliers
    actual, predicted = filter_outliers(actual, predicted, threshold=threshold)

    # Check if the data has been filtered to empty
    if len(actual) == 0 or len(predicted) == 0:
        print(f"Filtered data is empty for Geohash {geohash} with sample size {sample_size} and threshold {threshold}.")
        return

    # Calculate baseline predictions using the mean of the actual values
    baseline_predicted = np.full_like(actual, np.mean(actual))

    # Ensure the sample size is not greater than the filtered data size
    sample_size = min(sample_size, len(actual))

    # Plot True Values, Predictions, and Baseline
    plt.figure(figsize=(20, 15))
    if plot_type == 'line':
        plt.plot(actual[:sample_size], label='True Values', linestyle='-', linewidth=marker_size)
        plt.plot(predicted[:sample_size], label='Predictions', linestyle='-', linewidth=marker_size, color=marker_color)
        plt.plot(baseline_predicted[:sample_size], label='Baseline', linestyle='--', linewidth=marker_size, color='green')
    elif plot_type == 'scatter':
        plt.scatter(range(sample_size), actual[:sample_size], label='True Values', c='blue', s=marker_size)
        plt.scatter(range(sample_size), predicted[:sample_size], label='Predictions', c=marker_color, s=marker_size)
        plt.scatter(range(sample_size), baseline_predicted[:sample_size], label='Baseline', c='green', s=marker_size)
    plt.legend()
    plt.xlabel('Sample Index')
    plt.ylabel('Time to Next Ride (s)')
    plt.title(f'True Values vs. Predictions vs. Baseline for Geohash {geohash} (Sample Size: {sample_size})')

    # Save or show the plot
    if save_path:
        plt.savefig(save_path)
    else:
        plt.show()
    plt.close()

    # Calculate percentage error for predictions and baseline
    percentage_error_model = 100 * abs((actual[:sample_size] - predicted[:sample_size]) / actual[:sample_size])
    percentage_error_baseline = 100 * abs((actual[:sample_size] - baseline_predicted[:sample_size]) / actual[:sample_size])

    # Plot Percentage Error for Predictions and Baseline
    plt.figure(figsize=(20, 15))
    if plot_type == 'line':
        plt.plot(percentage_error_model, label='Percentage Error (Model)', linestyle='-', linewidth=marker_size, color=marker_color)
        plt.plot(percentage_error_baseline, label='Percentage Error (Baseline)', linestyle='--', linewidth=marker_size, color='green')
    elif plot_type == 'scatter':
        plt.scatter(range(sample_size), percentage_error_model, label='Percentage Error (Model)', c=marker_color, s=marker_size)
        plt.scatter(range(sample_size), percentage_error_baseline, label='Percentage Error (Baseline)', c='green', s=marker_size)
    plt.legend()
    plt.xlabel('Sample Index')
    plt.ylabel('Percentage Error (%)')
    plt.title(f'Percentage Error (Model vs. Baseline) for Geohash {geohash} (Sample Size: {sample_size})')

    # Save or show the plot
    if save_path:
        plt.savefig(save_path.replace('.png', '_error.png'))
    else:
        plt.show()
    plt.close()

In [ ]:
# Saving function to mass process evaluations
def save_geohash_analysis(predicted_vals, geohashes, plot_type='line', marker_color='blue', marker_size=10):
    sample_sizes = [100, 500, 1000]  # Define sample sizes ----------
    thresholds = [3000, 2500, 2000]  # Define thresholds -----------

    for geohash in geohashes:
        if geohash not in predicted_vals:
            print(f"Geohash {geohash} not found in the predictions dictionary.")
            continue

        # Create directory for the geohash
        directory = f"./{geohash}" # Add path ---------
        os.makedirs(directory, exist_ok=True)

        metrics_list = []

        for sample_size in sample_sizes:
            for threshold in thresholds:
                plot_filename = f"{directory}/{geohash}_plot_samplesize_{sample_size}_threshold_{threshold}.png"
                plot_filtered_results_with_baseline(
                    predicted_vals=predicted_vals,
                    geohash=geohash,
                    sample_size=sample_size,
                    threshold=threshold,
                    plot_type=plot_type,
                    marker_color=marker_color,
                    marker_size=marker_size,
                    save_path=plot_filename
                )

                metrics = calculate_performance_metrics(predicted_vals, geohash)

                if metrics is not None:
                    mae_model, mse_model, rmse_model, mae_baseline, mse_baseline, rmse_baseline = metrics

                    # Store the metrics
                    metrics_list.append({
                        'Geohash': geohash,
                        'Sample Size': sample_size,
                        'Threshold': threshold,
                        'Model MAE': mae_model,
                        'Baseline MAE': mae_baseline,
                        'Model MSE': mse_model,
                        'Baseline MSE': mse_baseline,
                        'Model RMSE': rmse_model,
                        'Baseline RMSE': rmse_baseline,
                    })

        # Save metrics to a CSV file
        metrics_df = pd.DataFrame(metrics_list)
        metrics_filename = f"{directory}/{geohash}_performance_metrics.csv"
        metrics_df.to_csv(metrics_filename, index=False)

Function Usage, all parameters can be adjusted

In [ ]:
plot_results(predicted_vals, 'dr5rus', 500, plot_type='line', marker_color='blue', marker_size=3)

In [ ]:
plot_filtered_results_with_baseline(predicted_vals, 'dr5ru7', sample_size=400,
                                    plot_type='line', marker_color='red', marker_size=2,
                                    threshold=2000, percentage=None)

In [ ]:
calculate_performance_metrics(predicted_vals, 'dr5ru7', threshold = None, percentage = None)

In [ ]:
geohashes = ['dr5rus', 'dr5ru6', 'dr5ru4', 'dr5ruk', 'dr5rvp', 'dr5ru9',
       'dr5rud', 'dr5rvj', 'dr72h8', 'dr5ru2', 'dr5rsk', 'dr5rue',
       'dr5rgb', 'dr5ru7', 'dr5rum', 'dr5ru0', 'dr5ref', 'dr5rsp',
       'dr5rvn', 'dr5rsq', 'dr72j0', 'dr5ruq', 'dr5rsr', 'dr5ru5',
       'dr5ruu', 'dr5rez', 'dr5ru3', 'dr5rsj', 'dr5reg', 'dr5reu',
       'dr5rsn', 'dr5rsh', 'dr5ru1', 'dr5rsm', 'dr5ru8', 'dr5rug']  # List of geohashes to analyze

save_geohash_analysis(predicted_vals, geohashes, plot_type='line', marker_color='red', marker_size=2)